## Explore the OpenFDA API Structure

In [ ]:
import requests

response = requests.get(
    "https://api.fda.gov/drug/label.json",
    params={"search": "indications_and_usage:diabetes", "limit": 1}
)
data = response.json()

print(data.keys())
print("---")
print(data["results"][0].keys()) # first result, which is a dict of drug label info

dict_keys(['meta', 'results'])
---
dict_keys(['spl_product_data_elements', 'indications_and_usage', 'dosage_and_administration', 'dosage_forms_and_strengths', 'contraindications', 'warnings_and_cautions', 'adverse_reactions', 'adverse_reactions_table', 'drug_interactions', 'use_in_specific_populations', 'use_in_specific_populations_table', 'pregnancy', 'pediatric_use', 'pediatric_use_table', 'geriatric_use', 'overdosage', 'description', 'clinical_pharmacology', 'mechanism_of_action', 'pharmacodynamics', 'pharmacokinetics', 'nonclinical_toxicology', 'carcinogenesis_and_mutagenesis_and_impairment_of_fertility', 'clinical_studies', 'clinical_studies_table', 'how_supplied', 'information_for_patients', 'spl_unclassified_section', 'package_label_principal_display_panel', 'set_id', 'id', 'effective_time', 'version', 'openfda'])


## Inspect the openfda Sub-Object

In [3]:
first_result = data["results"][0]
print(first_result["openfda"].keys())
print("---")
print("Brand name:", first_result["openfda"].get("brand_name"))
print("Generic name:", first_result["openfda"].get("generic_name"))
print("Drug class:", first_result["openfda"].get("pharm_class_epc"))
print("manufacturer_name form:", first_result["openfda"].get("manufacturer_name"))

dict_keys(['application_number', 'brand_name', 'generic_name', 'manufacturer_name', 'product_ndc', 'product_type', 'route', 'substance_name', 'rxcui', 'spl_id', 'spl_set_id', 'package_ndc', 'original_packager_product_ndc', 'nui', 'pharm_class_epc', 'pharm_class_cs', 'unii'])
---
Brand name: ['Glimepiride']
Generic name: ['GLIMEPIRIDE']
Drug class: ['Sulfonylurea [EPC]']
manufacturer_name form: ['American Health Packaging']


## Define the DrugRecord Data Model

In [4]:
from pydantic import BaseModel
from typing import List, Optional


class DrugRecord(BaseModel):
    brand_name: str
    generic_name: str
    drug_class: Optional[str] = None
    indications_and_usage: str
    dosage_and_administration: Optional[str] = None
    contraindications: Optional[str] = None
    warnings_and_cautions: Optional[str] = None
    adverse_reactions: Optional[str] = None
    drug_interactions: Optional[str] = None
    mechanism_of_action: Optional[str] = None
    topic: str
    source: str = "openfda"

In [5]:
test_drug = DrugRecord(
    brand_name=first_result["openfda"].get("brand_name", ["Unknown"])[0],
    generic_name=first_result["openfda"].get("generic_name", ["Unknown"])[0],
    drug_class=first_result["openfda"].get("pharm_class_epc", [None])[0],
    indications_and_usage=" ".join(first_result.get("indications_and_usage", [])),
    dosage_and_administration=" ".join(first_result.get("dosage_and_administration", [])) or None,
    contraindications=" ".join(first_result.get("contraindications", [])) or None,
    warnings_and_cautions=" ".join(first_result.get("warnings_and_cautions", [])) or None,
    adverse_reactions=" ".join(first_result.get("adverse_reactions", [])) or None,
    drug_interactions=" ".join(first_result.get("drug_interactions", [])) or None,
    mechanism_of_action=" ".join(first_result.get("mechanism_of_action", [])) or None,
    topic="diabetes",
)

print(test_drug)

brand_name='Glimepiride' generic_name='GLIMEPIRIDE' drug_class='Sulfonylurea [EPC]' indications_and_usage='1 INDICATIONS AND USAGE Glimepiride tablets are indicated as an adjunct to diet and exercise to improve glycemic control in adults with type 2 diabetes mellitus [see Clinical Studies (14.1) ]. Limitations of Use Glimepiride tablets should not be used for the treatment of type 1 diabetes mellitus or diabetic ketoacidosis, as it would not be effective in these settings. Glimepiride tablets are a sulfonylurea indicated as an adjunct to diet and exercise to improve glycemic control in adults with type 2 diabetes mellitus ( 1 ). Limitations of Use: Not for treating type 1 diabetes mellitus or diabetic ketoacidosis ( 1 ).' dosage_and_administration='2 DOSAGE AND ADMINISTRATION Recommended starting dose is 1 or 2 mg once daily. Increase in 1 or 2 mg increments no more frequently than every 1 to 2 weeks based on glycemic response. Maximum recommended dose is 8 mg once daily ( 2.1 ). Admin

## Search + Fetch Function

In [ ]:
import time

def fetch_drugs_raw(topic: str, limit: int = 150, max_retries: int = 3):
    """
    Query OpenFDA for drugs whose indications mention this topic.
    Returns raw JSON results (list of drug label dicts).
    Retries on timeout/connection errors; returns empty list if all retries fail.
    """
    for attempt in range(max_retries):
        try:
            response = requests.get(
                "https://api.fda.gov/drug/label.json",
                params={"search": f"indications_and_usage:{topic}", "limit": limit},
                timeout=15
            )
            if response.status_code != 200:
                return []
            data = response.json()
            return data.get("results", [])
        
        except requests.exceptions.RequestException as e:
            print(f"  Attempt {attempt + 1}/{max_retries} failed for '{topic}': {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                print(f"  Giving up on '{topic}' after {max_retries} attempts")
                return []


# Test with a small limit first
test_results = fetch_drugs_raw("diabetes", limit=5)
print(f"Found {len(test_results)} drug records")
print("Brand names:", [r["openfda"].get("brand_name", ["Unknown"])[0] for r in test_results]) # [0] to get the first brand name if multiple are present

Found 5 drug records
Brand names: ['Glimepiride', 'Unknown', 'Unknown', 'Unknown', 'Unknown']


## Inspect an "Unknown" Record

In [ ]:
unknown_record = test_results[1]  # second result, one of the "Unknown" ones
print(unknown_record["openfda"]) # no results for brand_name, generic_name

{}


## Handle Missing openfda Data Safely

In [ ]:
def parse_drug_record(raw: dict, topic: str) -> DrugRecord:
    """
    Parse one raw OpenFDA result into a DrugRecord.
    Raises ValueError if the record lacks identifiable drug name data.
    """
    openfda = raw.get("openfda", {})

    # Extract brand_name and generic_name, they must be present in at least one form to identify the drug    
    brand_name = openfda.get("brand_name", [None])[0]
    generic_name = openfda.get("generic_name", [None])[0]
    
    if not brand_name and not generic_name:
        raise ValueError("No brand_name or generic_name — cannot identify drug")
    
    return DrugRecord(
        brand_name=brand_name or generic_name,
        generic_name=generic_name or brand_name,
        drug_class=openfda.get("pharm_class_epc", [None])[0],
        indications_and_usage=" ".join(raw.get("indications_and_usage", [])),
        dosage_and_administration=" ".join(raw.get("dosage_and_administration", [])) or None,
        contraindications=" ".join(raw.get("contraindications", [])) or None,
        warnings_and_cautions=" ".join(raw.get("warnings_and_cautions", [])) or None,
        adverse_reactions=" ".join(raw.get("adverse_reactions", [])) or None,
        drug_interactions=" ".join(raw.get("drug_interactions", [])) or None,
        mechanism_of_action=" ".join(raw.get("mechanism_of_action", [])) or None,
        topic=topic,
    )


def parse_drugs_safe(raw_results: list, topic: str):
    """Parse a batch, skipping and logging any that fail (e.g., missing drug identity)."""
    parsed = []
    failed = []
    
    for raw in raw_results:
        try:
            drug = parse_drug_record(raw, topic=topic)
            parsed.append(drug)
        except Exception as e:
            failed.append({"error": str(e)})
    
    return parsed, failed


# Test on our 5 results from Cell 4
parsed_drugs, failed_drugs = parse_drugs_safe(test_results, topic="diabetes")
print(f"Parsed: {len(parsed_drugs)}")
print(f"Failed: {len(failed_drugs)}")
print("Parsed brand names:", [d.brand_name for d in parsed_drugs])

Parsed: 1
Failed: 4
Parsed brand names: ['Glimepiride']


## Add Deduplication (Avoid Same Drug Twice)

In [28]:
def dedupe_drugs(drugs: list) -> list:
    """Keep only the first occurrence of each brand_name."""
    seen = set()
    deduped = []
    for drug in drugs:
        key = drug.brand_name.lower().strip()
        if key not in seen:
            seen.add(key)
            deduped.append(drug)
    return deduped


# Test with a larger fetch to see real numbers
test_results_larger = fetch_drugs_raw("diabetes", limit=125)
parsed_larger, failed_larger = parse_drugs_safe(test_results_larger, topic="diabetes")
deduped_larger = dedupe_drugs(parsed_larger)

print(f"Raw fetched: {len(test_results_larger)}")
print(f"Parsed (usable): {len(parsed_larger)}")
print(f"Failed (no identity): {len(failed_larger)}")
print(f"After dedup: {len(deduped_larger)}")
print("Brand names:", [d.brand_name for d in deduped_larger])

Raw fetched: 125
Parsed (usable): 50
Failed (no identity): 75
After dedup: 28
Brand names: ['Glimepiride', 'Amlodipine Besylate', 'Verapamil Hydrochloride', 'Glipizide', 'SIMVASTATIN', 'ZITUVIMET', 'CHOLESTYRAMINE LIGHT', 'Atorvastatin calcium', 'Lisinopril and Hydrochlorothiazide', 'Fenofibrate', 'MEB Gluco-Vitality Patch', 'Metoprolol Tartrate', 'Metformin Hydrochloride', 'COLESEVELAM HYDROCHLORIDE', 'olmesartan medoxomil-hydrochlorothiazide', 'MEB Puri-Mega', 'Metoprolol Succinate', 'Benazepril Hydrochloride', 'FARXIGA', 'Lisinopril', 'Losartan Potassium', 'Lovastatin', 'Lopressor', 'Irbesartan', 'Phentermine Hydrochloride', 'Amlodipine and Olmesartan Medoxomil', 'PIOGLITAZONE HYDROCHLORIDE', 'Omega-3-acid ethyl esters']


## Cap at 25 and Finalize the Fetch Function

In [ ]:
def fetch_drugs_for_topic(topic: str, raw_limit: int = 150, target_count: int = 25):
    """
    Full OpenFDA pipeline for one topic: fetch -> parse safely -> dedupe -> cap.
    """
    raw_results = fetch_drugs_raw(topic, limit=raw_limit)
    parsed, failed = parse_drugs_safe(raw_results, topic=topic)
    deduped = dedupe_drugs(parsed)
    capped = deduped[:target_count] # top 25 usable drugs for the topic
    
    return capped, failed


# Test again with the finalized function
final_drugs, failed = fetch_drugs_for_topic("diabetes")
print(f"Final usable drugs: {len(final_drugs)}")
print(f"Failed (no identity): {len(failed)}")
print("Brand names:", [d.brand_name for d in final_drugs])

Final usable drugs: 25
Failed (no identity): 84
Brand names: ['Glimepiride', 'Amlodipine Besylate', 'Verapamil Hydrochloride', 'Glipizide', 'SIMVASTATIN', 'ZITUVIMET', 'CHOLESTYRAMINE LIGHT', 'Atorvastatin calcium', 'Lisinopril and Hydrochlorothiazide', 'Fenofibrate', 'MEB Gluco-Vitality Patch', 'Metoprolol Tartrate', 'Metformin Hydrochloride', 'COLESEVELAM HYDROCHLORIDE', 'olmesartan medoxomil-hydrochlorothiazide', 'MEB Puri-Mega', 'Metoprolol Succinate', 'Benazepril Hydrochloride', 'FARXIGA', 'Lisinopril', 'Losartan Potassium', 'Lovastatin', 'Lopressor', 'Irbesartan', 'Phentermine Hydrochloride']


## Save to Disk (Same JSONL Pattern as Phase 2)

In [ ]:
from pathlib import Path

def save_drugs(drugs: list, topic: str, output_dir: str = "../data/raw/openfda"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    filepath = Path(output_dir) / f"{topic}.jsonl"
    
    with open(filepath, "w", encoding="utf-8") as f:  # "w" not "a" — full overwrite is fine here since we always fetch fresh + cap at 25
        for drug in drugs:
            f.write(drug.model_dump_json() + "\n")
    
    return filepath


saved_path = save_drugs(final_drugs, topic="diabetes")
print("Saved to:", saved_path.resolve())

Saved to: C:\Users\DELL\Desktop\medrag\data\raw\openfda\diabetes.jsonl


## Run the Full Batch — All 36 Topics

In [31]:
import time

topics = [
    "diabetes", "hypertension", "obesity", "hyperlipidemia",
    "asthma", "copd", "pneumonia", "tuberculosis",
    "coronary artery disease", "heart failure", "stroke", "arrhythmia",
    "malaria", "dengue fever", "hiv aids", "hepatitis b", "covid-19", "typhoid",
    "depression", "anxiety disorder",
    "peptic ulcer disease", "irritable bowel syndrome", "hepatitis c",
    "osteoarthritis", "rheumatoid arthritis", "osteoporosis",
    "hypothyroidism", "hyperthyroidism",
    "epilepsy", "migraine", "parkinson's disease",
    "chronic kidney disease",
    "breast cancer", "lung cancer",
    "anemia in pregnancy", "malnutrition"
]

all_drug_results = []

for topic in topics:  # reuse the same 36-topic list from Phase 2
    drugs, failed = fetch_drugs_for_topic(topic)
    save_drugs(drugs, topic=topic)
    all_drug_results.append({"topic": topic, "saved": len(drugs), "failed": len(failed)})
    print(f"{topic}: saved {len(drugs)}, failed {len(failed)}")
    time.sleep(0.3)  # basic rate limiting between topics

total_saved = sum(r["saved"] for r in all_drug_results)
total_failed = sum(r["failed"] for r in all_drug_results)
print(f"\n=== DONE === Total saved: {total_saved}, Total failed (no identity): {total_failed}")

diabetes: saved 25, failed 84
hypertension: saved 25, failed 93
obesity: saved 14, failed 88
hyperlipidemia: saved 25, failed 84
asthma: saved 25, failed 86
copd: saved 25, failed 69
  Attempt 1/3 failed for 'pneumonia': HTTPSConnectionPool(host='api.fda.gov', port=443): Max retries exceeded with url: /drug/label.json?search=indications_and_usage%3Apneumonia&limit=150 (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.fda.gov', port=443) at 0x1dc872e3790>, 'Connection to api.fda.gov timed out. (connect timeout=15)'))
  Attempt 2/3 failed for 'pneumonia': HTTPSConnectionPool(host='api.fda.gov', port=443): Max retries exceeded with url: /drug/label.json?search=indications_and_usage%3Apneumonia&limit=150 (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.fda.gov', port=443) at 0x1dc877d13d0>, 'Connection to api.fda.gov timed out. (connect timeout=15)'))
pneumonia: saved 19, failed 113
tuberculosis: saved 25, failed 94
coronary artery disease: saved 25, failed 95
heart failur